In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingRegressor

PROC, GRID = Path("../data/processed"), 0.25
T0, T1 = 2013 * 12 + 0, 2026 * 12 + 9      # month index t = year*12 + (month-1): Jan 2013 .. Oct 2026
REGION = "algeria"                          # we run the other regions later
OUTAGE = ["2017-12", "2018-10", "2022-04", "2022-07", "2022-08", "2024-01", "2024-02", "2024-06",
          "2024-07", "2024-11", "2025-12", "2026-01", "2026-03", "2026-04", "2026-07"] if REGION == "algeria" else []

def fit_loglog_smear(t):
    x, y = np.log(t.modis + 1), np.log(t.viirs + 1)
    b, a = np.polyfit(x, y, 1)
    return b, a, np.mean(np.exp(y - (a + b * x)))

def pred_loglog_smear(p, m):
    b, a, smear = p
    return smear * np.exp(a + b * np.log(m + 1)) - 1

# 1. Table with one row per (square, month): MODIS count, VIIRS count, zeros filled in.
#    Why: the model must also learn the cases where nothing was seen.
def prep(g, name):
    t = g.year.astype("int64") * 12 + g.month.astype("int64") - 1
    g = g.assign(t=t, iy=np.floor(g.lat / GRID).astype("int64"), ix=np.floor(g.lon / GRID).astype("int64"))
    return g.set_index(["t", "iy", "ix"])["count"].rename(name)

out_t = [int(m[:4]) * 12 + int(m[5:7]) - 1 for m in OUTAGE]
months = np.setdiff1d(np.arange(T0, T1 + 1), out_t)
m_ = prep(pd.read_parquet(PROC / f"{REGION}_grid_m.parquet"), "modis")
v_ = prep(pd.read_parquet(PROC / f"{REGION}_grid_v.parquet"), "viirs")
m_, v_ = [s[s.index.get_level_values("t").isin(months)] for s in (m_, v_)]
cells = pd.concat([m_.reset_index()[["iy", "ix"]], v_.reset_index()[["iy", "ix"]]]).drop_duplicates()
cube = cells.merge(pd.DataFrame({"t": months}), how="cross")
cube = cube.join(m_, on=["t", "iy", "ix"]).join(v_, on=["t", "iy", "ix"])
cube[["modis", "viirs"]] = cube[["modis", "viirs"]].fillna(0)

# 2. Inputs, all from MODIS, so they will also exist for 2003-2012.
mod = cube.set_index(["t", "iy", "ix"])["modis"]
nb = np.zeros(len(cube))
for dy in (-1, 0, 1):
    for dx in (-1, 0, 1):
        if (dy, dx) != (0, 0):
            key = pd.MultiIndex.from_arrays([cube.t, cube.iy + dy, cube.ix + dx])
            nb += mod.reindex(key).fillna(0).to_numpy()
cube["modis_nb"] = nb                                                          # MODIS in the 8 neighbours
cube["cell_mean"] = cube.groupby(["iy", "ix"])["modis"].transform("mean")      # the square's usual MODIS level
cube["year"], cube["month"] = cube.t // 12, cube.t % 12 + 1
FEATS = ["modis", "modis_nb", "cell_mean", "month", "iy", "ix"]
tot = cube.groupby("t")[["modis", "viirs"]].sum()
print(len(cube), "square-months |", len(cells), "squares |", len(months), "months")

# 3. Leave-one-year-out: each year is predicted by a model that never saw it.
#    ML predicts each square directly. The baseline predicts the monthly total, then uses the share trick.
cube["ml"], cube["base"] = np.nan, np.nan
for y in sorted(cube.year.unique()):
    tr, te = cube.year != y, cube.year == y
    mdl = HistGradientBoostingRegressor(loss="poisson", max_iter=300, learning_rate=0.05, max_leaf_nodes=15,
                                        min_samples_leaf=50, l2_regularization=1.0, random_state=0)
    mdl.fit(cube.loc[tr, FEATS], cube.loc[tr, "viirs"])
    cube.loc[te, "ml"] = mdl.predict(cube.loc[te, FEATS])

    p = fit_loglog_smear(tot[tot.index // 12 != y])
    tte = tot[tot.index // 12 == y]
    share = cube.loc[te, "modis"] / cube.loc[te, "t"].map(tte.modis)
    cube.loc[te, "base"] = (share * cube.loc[te, "t"].map(pred_loglog_smear(p, tte.modis))).fillna(0)

# 4. Scores: monthly totals (what you saw before) and individual squares (the map).
res = cube.groupby("t")[["viirs", "ml", "base"]].sum()
res = res[res.viirs > 0]
def score(col):
    rel = (res.viirs - res[col]).abs() / res.viirs
    return {"median monthly-total error": f"{rel.median():.1%}",
            "total bias": f"{res[col].sum() / res.viirs.sum() - 1:+.1%}",
            "square MAE": round(float((cube.viirs - cube[col]).abs().mean()), 3),
            "square correlation": round(float(np.corrcoef(cube.viirs, cube[col])[0, 1]), 3)}
display(pd.DataFrame({"baseline + share trick": score("base"), "ML (per square)": score("ml")}))

yr = res.groupby(res.index // 12)[["viirs", "base", "ml"]].sum().round(0)
yr.index.name = "year"
yr["base err"] = (yr.base / yr.viirs - 1).map("{:+.0%}".format)
yr["ml err"] = (yr.ml / yr.viirs - 1).map("{:+.0%}".format)
display(yr)

# 5. How much of VIIRS sits where MODIS saw nothing that month? The share trick gives those squares 0.
zero = cube.modis == 0
print(f"{cube.viirs[zero].sum() / cube.viirs.sum():.1%} of VIIRS cell-days fall in squares where MODIS saw nothing that month")

59192 square-months | 392 squares | 151 months


,baseline + share trick,ML (per square)
median monthly-total error,16.0%,13.7%
total bias,-3.3%,-1.0%
square MAE,0.83,0.702
square correlation,0.842,0.869


,viirs,base,ml,base err,ml err
year,,,,,
2013,4163.0,3777.0,3931.0,-9%,-6%
2014,8403.0,6927.0,7486.0,-18%,-11%
2015,3846.0,3913.0,3763.0,+2%,-2%
2016,5412.0,5385.0,5668.0,-0%,+5%
2017,6933.0,6583.0,6891.0,-5%,-1%
2018,2116.0,1918.0,2152.0,-9%,+2%
2019,5157.0,4988.0,4961.0,-3%,-4%
2020,7400.0,6782.0,7273.0,-8%,-2%
2021,4760.0,4754.0,4563.0,-0%,-4%


30.9% of VIIRS cell-days fall in squares where MODIS saw nothing that month


In [3]:
TRAIN_FROM = 2016
tr = cube.year >= TRAIN_FROM      # the "known period": 2016-2026
te = cube.year <= 2015            # the "unknown past": 2013, 2014, 2015

# Option B: each square's usual MODIS level, computed from training years only
cm_train = cube[tr].groupby(["iy", "ix"])["modis"].mean().rename("cell_mean_tr")
cube = cube.drop(columns="cell_mean_tr", errors="ignore").join(cm_train, on=["iy", "ix"])

In [5]:
PARAMS = dict(loss="poisson", max_iter=300, learning_rate=0.05, max_leaf_nodes=15,
              min_samples_leaf=50, l2_regularization=1.0, random_state=0)

def fit_predict(feats):
    mdl = HistGradientBoostingRegressor(**PARAMS)
    mdl.fit(cube.loc[tr, feats], cube.loc[tr, "viirs"])
    return mdl.predict(cube.loc[te, feats])

FEATS_A = ["modis", "modis_nb", "cell_mean",    "month", "iy", "ix"]
FEATS_B = ["modis", "modis_nb", "cell_mean_tr", "month", "iy", "ix"]

cube["ml_A"], cube["ml_B"] = np.nan, np.nan
cube.loc[te, "ml_A"] = fit_predict(FEATS_A)
cube.loc[te, "ml_B"] = fit_predict(FEATS_B)

In [8]:
base_tot = cube.loc[te].groupby("t")["base_bt"].transform("sum")   # baseline monthly total
ml_tot   = cube.loc[te].groupby("t")["ml_B"].transform("sum")      # ML monthly total
cube["hyb"] = np.nan
cube.loc[te, "hyb"] = cube.loc[te, "ml_B"] / ml_tot * base_tot     # ML shape, baseline total

In [9]:
# baseline: fit log-log on training-year monthly totals, predict the 2013-2015 totals,
# then split each total by MODIS share
p    = fit_loglog_smear(tot[tot.index // 12 >= TRAIN_FROM])
tte  = tot[tot.index // 12 <= 2015]
share = cube.loc[te, "modis"] / cube.loc[te, "t"].map(tte.modis)
cube["base_bt"] = np.nan
cube.loc[te, "base_bt"] = (share * cube.loc[te, "t"].map(pred_loglog_smear(p, tte.modis))).fillna(0)

d = cube[te]
rows = {}
for name, col in [("baseline", "base_bt"), ("ML A (cell_mean all years)", "ml_A"),
                  ("ML B (cell_mean train only)", "ml_B"), ("hybrid", "hyb")]:
    m = d.groupby("t")[["viirs", col]].sum()
    m = m[m.viirs > 0]
    r = {f"bias {y} (%)": 100 * (d.loc[d.year == y, col].sum() / d.loc[d.year == y, "viirs"].sum() - 1)
         for y in (2013, 2014, 2015)}
    r["total bias (%)"] = 100 * (d[col].sum() / d.viirs.sum() - 1)
    r["median monthly error (%)"] = 100 * ((m.viirs - m[col]).abs() / m.viirs).median()
    r["square MAE"] = (d.viirs - d[col]).abs().mean()
    rows[name] = r
display(pd.DataFrame(rows).round(2))

,baseline,ML A (cell_mean all years),ML B (cell_mean train only),hybrid
bias 2013 (%),-10.86,-8.24,-10.18,-10.86
bias 2014 (%),-20.90,-14.71,-16.58,-20.90
bias 2015 (%),-1.06,-7.82,-9.22,-1.06
total bias (%),-13.70,-11.46,-13.23,-13.70
median monthly error (%),20.41,23.19,24.51,20.41
square MAE,0.86,0.73,0.74,0.76


In [10]:
# ---- Part A: is the baseline's bias a "busy months" problem? ----
d  = cube[te]
mt = d.groupby("t").agg(modis=("modis", "sum"), viirs=("viirs", "sum"), base=("base_bt", "sum"))
mt = mt[mt.viirs > 0].copy()
mt["year"]  = mt.index // 12
mt["level"] = pd.qcut(mt.modis, 3, labels=["quiet", "medium", "busy"])   # grouped by MODIS, not VIIRS

by_level = mt.groupby("level", observed=True).apply(
    lambda g: pd.Series({"months": len(g),
                         "bias (%)": 100 * (g.base.sum() / g.viirs.sum() - 1)}))
display(by_level.round(1))

by_year_level = mt.groupby(["year", "level"], observed=True).apply(
    lambda g: 100 * (g.base.sum() / g.viirs.sum() - 1)).unstack().round(1)
display(by_year_level)

# ---- Part B: does the MODIS-to-VIIRS relationship drift over time? ----
p_all = fit_loglog_smear(tot[tot.index // 12 >= TRAIN_FROM])      # curve fitted on 2016+ only
am = tot[tot.viirs > 0].copy()
am["pred"] = pred_loglog_smear(p_all, am.modis)
am["year"] = am.index // 12
yr = am.groupby("year")[["viirs", "pred"]].sum()
yr["bias (%)"] = 100 * (yr.pred / yr.viirs - 1)
display(yr.round(1))

,months,bias (%)
level,,
quiet,12.0,26.8
medium,12.0,19.2
busy,12.0,-20.2


level,quiet,medium,busy
year,,,
2013,52.8,18.4,-18.6
2014,8.2,20.6,-23.2
2015,19.0,19.2,-13.0


,viirs,pred,bias (%)
year,,,
2013,4163.0,3711.0,-10.9
2014,8403.0,6646.7,-20.9
2015,3846.0,3805.1,-1.1
2016,5412.0,5162.3,-4.6
2017,6933.0,6235.2,-10.1
2018,2116.0,1960.6,-7.3
2019,5157.0,4797.8,-7.0
2020,7400.0,6353.6,-14.1
2021,4760.0,4536.6,-4.7
